문서 파싱

In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day01" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w5" / "day01"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))


프로젝트 루트  : c:\hw\hw\python_basic\hanwha-agent


In [2]:
SAMPLES = SANDBOX / "samples"  # 실습 재료 배치할 폴더 

In [3]:
import shutil
CANDIDATES = [pathlib.Path.home() / 'Downloads' / 'seed_docs', ROOT.parent / 'seed_docs']
SAMPLES.mkdir(parents=True, exist_ok=True)
found = [p for p in SAMPLES.rglob('*') if p.is_file()]
if found:
    print(f'이미 있습니다 : {SAMPLES.relative_to(ROOT)}')
    print(f'파일 {len(found)}개')
else:
    SRC = next((c for c in CANDIDATES if c.is_dir()), None)
    if SRC is None:
        print('재료 폴더를 찾지 못했습니다. 아래 두 자리 중 한 곳에 풀어 주세요 —')
        for c in CANDIDATES:
            print('   ', c)
    else:
        shutil.copytree(SRC, SAMPLES, dirs_exist_ok=True)
        print(f'복사 완료 : {SAMPLES.relative_to(ROOT)}')
        print(f"파일 {sum((1 for p in SAMPLES.rglob('*') if p.is_file()))}개")


이미 있습니다 : sandbox\w5\day01\samples
파일 30개


In [4]:
from collections import Counter  # 항목별 개수를 세는 표준 라이브러리 

ORDER = [".docx", ".pdf", ".hwpx"]

# 파일들만 모아서 확장자(소문자변환) 기준으로 카운트 
counts = Counter(p.suffix.lower() for p in SAMPLES.rglob("*") if p.is_file())
others = sum(n for ext, n in counts.items() if ext not in ORDER)

print(f"파일 {sum(counts.values())}개")
print(" " + " , ".join(f"{ext} {counts[ext]}" for ext in ORDER) + f" , 그 밖 {others}")


파일 30개
 .docx 10 , .pdf 11 , .hwpx 3 , 그 밖 6


In [5]:
import unicodedata

fs = [p.name for p in SAMPLES.glob("*.pdf")]
print(len(fs), "개")

for n in fs[:3]:
    print(n, unicodedata.is_normalized("NFC", n))

10 개
DOC-FI-009_법인카드_사용_지침_v1.4.pdf True
DOC-HR-002_복무_규정_v3.1.pdf True
DOC-HR-014_국내출장_여비_규정_v1.0.pdf True


PDF

In [6]:
from pypdf import PdfReader

# Path.glob("패턴") : 폴더 안에서 이름이 패턴에 맞는 파일을 찾아 하나씩 내놓는 객체 
# next(...) : 조회된 파일들 중 첫번째 하나만 꺼낸다. 
PDF = next(SAMPLES.glob("DOC-HR-014_*_v2.0.pdf"))

reader = PdfReader(str(PDF)) # 파일 열기, 경로 문자열로 넘기기 
print("쪽수 : ", len(reader.pages))

쪽수 :  16


In [7]:
first = reader.pages[0].extract_text()

print(repr(first[:60])) # repr() : 화면에 안보이는 것까지 그대로 보여준다. \n, 앞뒤 공백 등 
print(first[:60])

'일반\n 국내출장 여비 규정\n v2.0\n 문서번호 DOC-HR-014\n 시행일 2025-07-01\n 소관부서 '
일반
 국내출장 여비 규정
 v2.0
 문서번호 DOC-HR-014
 시행일 2025-07-01
 소관부서 


In [8]:
total = 0 

for page_no, page in enumerate(reader.pages, 1):
    text = (page.extract_text() or "").strip()
    total += len(text)
    print(f"p.{page_no}  {len(text)}자")

print("-"*15)
print(f"{len(reader.pages)} 페이지, 합계 약 {round(total, -2):,} 자")

p.1  108자
p.2  231자
p.3  443자
p.4  272자
p.5  1070자
p.6  903자
p.7  1005자
p.8  1042자
p.9  961자
p.10  222자
p.11  281자
p.12  282자
p.13  310자
p.14  244자
p.15  211자
p.16  216자
---------------
16 페이지, 합계 약 7,800 자


In [9]:
SCAN = SAMPLES / "_formats" / "sample_scanned.pdf"

scan_reader = PdfReader(str(SCAN))
print("쪽수 : ", len(scan_reader.pages))

total = 0

for page_no, page in enumerate(scan_reader.pages, 1):
    text = (page.extract_text() or "").strip()
    total += len(text)
    print(f"p.{page_no}  {len(text)}자")

print("-"*15)
print(f"{len(scan_reader.pages)} 페이지, 합계 약 {round(total, -2):,} 자")

쪽수 :  1
p.1  0자
---------------
1 페이지, 합계 약 0 자


In [10]:
from app.core.logging import get_logger

log = get_logger('app.rag.local_patsers')

blocks = []
for page_no, page in enumerate(scan_reader.pages, 1):
    text = (page.extract_text() or '').strip()
    if not text:
        log.info('텍스트 없음 (스캔본으로 보임): %s p.%d, SCAN.name, page_no')
        continue
    blocks.append(text)
print(f'블록 {len(blocks)}개')

11:28:05 INFO     app.rag.local_patsers: 텍스트 없음 (스캔본으로 보임): %s p.%d, SCAN.name, page_no
블록 0개


In [ ]:
from app.rag.local_parsers import parse_pdf

doc = parse_pdf(PDF)

print(f'블록 {len(doc.blocks)} · page_count {doc.page_count} · table_count {doc.table_count}')
sixth = doc.blocks[5]   # 6페이지 확인
print(f'여섯 번째 블록 : kind={sixth.kind}  locator={sixth.locator}')
print()
print('※ ImportError 가 나면 Kernel ▸ Restart Kernel 뒤 맨 위부터 다시 실행하세요.')

블록 16 · page_count 16 · table_count 0
여섯 번째 블록 : kind=조항  locator=p.6

※ ImportError 가 나면 Kernel ▸ Restart Kernel 뒤 맨 위부터 다시 실행하세요.


In [ ]:
# 스캔 PDF도 테스트 -> 로그 나오는지 확인하기
doc = parse_pdf(SCAN)
print(f"블럭: {len(doc.blocks)} / page_count : {doc.page_count} / table_count : {doc.table_count}")

11:57:56 INFO     app.rag.local_parsers: 텍스트 없음 (스캔본으로 보임): sample_scanned.pdf p.1
블럭: 0 / page_count : 1 / table_count : 0


DOCX 파싱

In [ ]:
import zipfile   # 압축파일을 여는 표준 라이브러리

# 워드 문서 조회 
DOCX = next(SAMPLES.glob('DOC-HR-014_*_v2.0.docx'))
print('첫 4바이트 :', DOCX.read_bytes()[:4], '  ← ZIP 표식')   # b'PK' zip 파일

# 압축 해제
with zipfile.ZipFile(DOCX) as zf:
    names = zf.namelist()     # 압축 안에 든 파일 이름 목록 추출
print(f'엔트리 {len(names)}개')

for name in names[:6]:
    mark = '  ← 본문이 여기 있습니다' if name == 'word/document.xml' else ''
    print(f'  {name}{mark}')

첫 4바이트 : b'PK\x03\x04'   ← ZIP 표식
엔트리 17개
  [Content_Types].xml
  _rels/.rels
  docProps/core.xml
  docProps/app.xml
  word/document.xml  ← 본문이 여기 있습니다
  word/_rels/document.xml.rels


In [14]:
from docx import Document

doc = Document(str(DOCX))

print(f"문단 : {len(doc.paragraphs)}개")
print(f"표 : {len(doc.tables)}개")
print("첫 문단 : ", doc.paragraphs[0].text)


문단 : 246개
표 : 7개
첫 문단 :  일반


*빈 문단을 빼고, 진짜 글자가 있는 문단만 남기기*

In [16]:
texts = [p.text.strip() for p in doc.paragraphs if p.text.strip()]

print(f"글자가 있는 문단 개수 : {len(texts)}개")
print(f"빈 문단 : {len(doc.paragraphs) - len(texts)}개")
print(texts[:3])

글자가 있는 문단 개수 : 226개
빈 문단 : 20개
['일반', '국내출장 여비 규정', 'v2.0']
